In [ ]:
import pandas as pd
import numpy as np
import os

def clear_screen():
    """Clears the console screen for better readability."""
    os.system('cls' if os.name == 'nt' else 'clear')

def load_and_clean_data(filename=r"C:\Users\Admin\Downloads\AI SHIKSHAVERTEX\archive (2).zip"):
    """Loads Netflix dataset and applies required cleaning tasks plus enhancements."""
    try:
        df = pd.read_csv(filename)
    except FileNotFoundError:
        print(f"Error: {filename} not found. Please verify the file path.")
        return None

    # Required Data Cleaning Tasks
    df['country'] = df['country'].fillna("Unknown")
    df['director'] = df['director'].fillna("Not Listed")
    df['cast'] = df['cast'].fillna("Not Listed")
    df['rating'] = df['rating'].fillna("Not Rated")
    df = df.dropna(subset=['date_added'])
    
    df['date_added'] = df['date_added'].str.strip()
    df['year_added'] = pd.to_datetime(df['date_added'], format='mixed').dt.year.astype('Int64')
    
    # Safely extract digits using regex for duration analysis
    duration_digits = df['duration'].astype(str).str.extract(r'(\d+)')[0]
    df['duration_num'] = np.where(
        df['type'] == 'Movie',
        pd.to_numeric(duration_digits, errors='coerce'),
        np.nan
    )
    
    return df

def print_formatted_table(series, col1_name, col2_name):
    """Outputs formatted ASCII tables in the console."""
    print(f"\n{col1_name:<35} | {col2_name:<10}")
    print("-" * 48)
    for index, value in series.items():
        print(f"{str(index)[:33]:<35} | {value:<10}")

def main():
    df = load_and_clean_data()
    if df is None:
        return

    while True:
        print("\n" + "="*55)
        print("             🎬 NETFLIX CONTENT EXPLORER 🎬")
        print("="*55)
        print("--- Analytics & Exploration Tools ---")
        print("1. Count of Movies vs TV Shows")
        print("2. Number of titles added each year")
        print("3. Top 10 producing countries")
        print("4. Top 5 most common genres")
        print("5. Rating distribution")
        print("6. Oldest and newest release years")
        print("7. Search titles by keyword")
        print("8. Content added in a specific year")
        print("9. Total unique countries")
        print("10. Summary statistics on release year")
        print("11. Top 5 Directors with the most content")
        print("12. Top 5 Longest Movies (by duration)")
        print("13. Advanced Search (Genre + Year Added)")
        print("14. Quit")
        
        choice = input("\nEnter your choice (1-14): ")
        clear_screen()
        
        if choice == '1':
            counts = df['type'].value_counts()
            print_formatted_table(counts, "Content Type", "Count")
            
        elif choice == '2':
            yearly_adds = df['year_added'].value_counts().sort_index(ascending=False)
            print_formatted_table(yearly_adds.head(10), "Year Added (Top 10 Recent)", "Titles")
            
        elif choice == '3':
            countries = df['country'].str.split(', ').explode().value_counts().head(10)
            print_formatted_table(countries, "Country", "Titles Produced")
            
        elif choice == '4':
            genres = df['listed_in'].str.split(', ').explode().value_counts().head(5)
            print_formatted_table(genres, "Genre", "Count")
            
        elif choice == '5':
            ratings = df['rating'].value_counts()
            print_formatted_table(ratings, "Rating", "Count")
            
        elif choice == '6':
            oldest = df['release_year'].min()
            newest = df['release_year'].max()
            print(f"\nOldest release year : {oldest}")
            print(f"Newest release year : {newest}")
            
        elif choice == '7':
            keyword = input("Enter keyword to search: ")
            results = df[df['title'].str.contains(keyword, case=False, na=False)]
            print(f"\nFound {len(results)} titles matching '{keyword}':")
            print(f"{'Type':<10} | {'Title':<45} | {'Release'}")
            print("-" * 70)
            for _, row in results.head(10).iterrows():
                print(f"{row['type']:<10} | {row['title'][:43]:<45} | {row['release_year']}")
            
        elif choice == '8':
            try:
                year_input = int(input("Enter year (e.g., 2020): "))
                results = df[df['year_added'] == year_input]
                print(f"\nTotal content added in {year_input}: {len(results)} titles")
                print(f"{'Type':<10} | {'Title':<50}")
                print("-" * 65)
                for _, row in results.head(8).iterrows():
                    print(f"{row['type']:<10} | {row['title'][:48]:<50}")
            except ValueError:
                print("Invalid year format. Please enter a number.")
                
        elif choice == '9':
            unique_countries = df['country'].str.split(', ').explode().replace('Unknown', np.nan).nunique()
            print(f"\nTotal unique countries producing content: {unique_countries}")
            
        elif choice == '10':
            years = df['release_year'].dropna()
            mean_yr = np.mean(years)
            median_yr = np.median(years)
            std_yr = np.std(years)
            q1 = np.percentile(years, 25)
            q3 = np.percentile(years, 75)
            
            print("\nEnhanced Summary Statistics on Release Year:")
            print("-" * 45)
            print(f"{'Statistic':<25} | {'Value':<10}")
            print("-" * 45)
            print(f"{'Mean':<25} | {mean_yr:.1f}")
            print(f"{'Median':<25} | {median_yr:.0f}")
            print(f"{'Std Deviation':<25} | {std_yr:.2f}")
            print(f"{'25th Percentile (Q1)':<25} | {q1:.0f}")
            print(f"{'75th Percentile (Q3)':<25} | {q3:.0f}")

        elif choice == '11':
            valid_directors = df[df['director'] != 'Not Listed']
            directors = valid_directors['director'].str.split(', ').explode().value_counts().head(5)
            print_formatted_table(directors, "Director", "Titles")
            
        elif choice == '12':
            top_movies = df.nlargest(5, 'duration_num')[['title', 'duration_num']]
            print(f"\n{'Movie Title':<45} | {'Duration (mins)'}")
            print("-" * 65)
            for _, row in top_movies.iterrows():
                print(f"{row['title'][:43]:<45} | {row['duration_num']:.0f}")

        elif choice == '13':
            genre_input = input("Enter a genre (e.g., Comedies, Dramas): ")
            try:
                year_input = int(input("Enter year added (e.g., 2019): "))
                results = df[(df['listed_in'].str.contains(genre_input, case=False, na=False)) & 
                             (df['year_added'] == year_input)]
                print(f"\nFound {len(results)} '{genre_input}' titles added in {year_input}:")
                print(f"{'Type':<10} | {'Title':<50}")
                print("-" * 65)
                for _, row in results.head(10).iterrows():
                    print(f"{row['type']:<10} | {row['title'][:48]:<50}")
            except ValueError:
                print("Invalid year format.")

        elif choice == '14':
            print("\nExiting Netflix Content Explorer. Goodbye!")
            break
            
        else:
            print("Invalid choice. Please select a valid option from 1-14.")

if __name__ == "__main__":
    main()


             🎬 NETFLIX CONTENT EXPLORER 🎬
--- Analytics & Exploration Tools ---
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year
11. Top 5 Directors with the most content
12. Top 5 Longest Movies (by duration)
13. Advanced Search (Genre + Year Added)
14. Quit



Enter your choice (1-14):  1



Content Type                        | Count     
------------------------------------------------
Movie                               | 6131      
TV Show                             | 2666      

             🎬 NETFLIX CONTENT EXPLORER 🎬
--- Analytics & Exploration Tools ---
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year
11. Top 5 Directors with the most content
12. Top 5 Longest Movies (by duration)
13. Advanced Search (Genre + Year Added)
14. Quit



Enter your choice (1-14):  2



Year Added (Top 10 Recent)          | Titles    
------------------------------------------------
2021                                | 1498      
2020                                | 1879      
2019                                | 2016      
2018                                | 1649      
2017                                | 1188      
2016                                | 429       
2015                                | 82        
2014                                | 24        
2013                                | 11        
2012                                | 3         

             🎬 NETFLIX CONTENT EXPLORER 🎬
--- Analytics & Exploration Tools ---
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year
11. Top 5 Director

...


Enter your choice (1-14):  11



Director                            | Titles    
------------------------------------------------
Rajiv Chilaka                       | 22        
Jan Suter                           | 21        
Raúl Campos                         | 19        
Suhas Kadav                         | 16        
Marcus Raboy                        | 16        

             🎬 NETFLIX CONTENT EXPLORER 🎬
--- Analytics & Exploration Tools ---
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year
11. Top 5 Directors with the most content
12. Top 5 Longest Movies (by duration)
13. Advanced Search (Genre + Year Added)
14. Quit



Enter your choice (1-14):  12



Movie Title                                   | Duration (mins)
-----------------------------------------------------------------
Black Mirror: Bandersnatch                    | 312
Headspace: Unwind Your Mind                   | 273
The School of Mischief                        | 253
No Longer kids                                | 237
Lock Your Girls In                            | 233

             🎬 NETFLIX CONTENT EXPLORER 🎬
--- Analytics & Exploration Tools ---
1. Count of Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Content added in a specific year
9. Total unique countries
10. Summary statistics on release year
11. Top 5 Directors with the most content
12. Top 5 Longest Movies (by duration)
13. Advanced Search (Genre + Year Added)
14. Quit



Enter your choice (1-14):  13
Enter a genre (e.g., Comedies, Dramas):  14
